In [1]:
!pip install pandas numpy scikit-learn joblib rdkit

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os
import numpy as np
import pandas as pd

from pathlib import Path

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [11]:
from pathlib import Path

# Datasets are in the same folder as the notebook
repo_root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "datasets" / "processed").is_dir()), None)
if repo_root is None:
    raise FileNotFoundError("Could not locate the repository root containing datasets/processed.")
data_dir = repo_root / "datasets" / "processed"

datasets = {
    "ALMANAC": data_dir / "almanac_final_without_last_score.xls",

    "DrugComb": data_dir / "DrugComb_LINCS_PRISM_final_without_last_score.zip",

    "DrugCombDB": data_dir / "DrugCombDBLINCS_PRISM_final_without_last_score.csv",

    "OncologyScreen": data_dir / "OncologyScreenLINCS_PRISM_final_without_last_score.xls",

    "Oneil": data_dir / "OneilLINCS_PRISM_final_without_last_score.csv"
}

# Check all five paths
print("Jupyter working directory:", data_dir)
print("\nChecking datasets:\n")

for name, path in datasets.items():
    print(f"{name}: {'FOUND' if path.exists() else 'NOT FOUND'}")

Jupyter working directory: C:\Users\sahas\Project FINAL

Checking datasets:

ALMANAC: FOUND
DrugComb: FOUND
DrugCombDB: FOUND
OncologyScreen: FOUND
Oneil: FOUND


In [12]:
morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)


def generate_fingerprint(smiles):

    if pd.isna(smiles):
        return None

    mol = Chem.MolFromSmiles(str(smiles).strip())

    if mol is None:
        return None

    fp = morgan_generator.GetFingerprint(mol)

    arr = np.zeros(2048, dtype=np.int8)

    DataStructs.ConvertToNumpyArray(fp, arr)

    return arr


print("Morgan fingerprint generator created successfully!")

Morgan fingerprint generator created successfully!


In [13]:
for name, path in datasets.items():

    print("\n" + "=" * 60)
    print("DATASET:", name)
    print("=" * 60)

    df = pd.read_csv(path)

    print("Shape:", df.shape)

    print("Columns:", df.columns.tolist())

    print("Missing values:", df.isnull().sum().sum())

    print("\nFirst five rows:")

    display(df.head())


DATASET: ALMANAC
Shape: (29251, 334)
Columns: ['DRS_feature_1', 'DRS_feature_2', 'DRS_feature_3', 'DRS_feature_4', 'DRS_feature_5', 'DRS_feature_6', 'DRS_feature_7', 'DRS_feature_8', 'DRS_feature_9', 'DRS_feature_10', 'DRS_feature_11', 'DRS_feature_12', 'DRS_feature_13', 'DRS_feature_14', 'DRS_feature_15', 'DRS_feature_16', 'DRS_feature_17', 'DRS_feature_18', 'DRS_feature_19', 'DRS_feature_20', 'DRS_feature_21', 'DRS_feature_22', 'DRS_feature_23', 'DRS_feature_24', 'DRS_feature_25', 'DRS_feature_26', 'DRS_feature_27', 'DRS_feature_28', 'DRS_feature_29', 'DRS_feature_30', 'DRS_feature_31', 'DRS_feature_32', 'DRS_feature_33', 'DRS_feature_34', 'DRS_feature_35', 'DRS_feature_36', 'DRS_feature_37', 'DRS_feature_38', 'DRS_feature_39', 'DRS_feature_40', 'DRS_feature_41', 'DRS_feature_42', 'DRS_feature_43', 'DRS_feature_44', 'DRS_feature_45', 'DRS_feature_46', 'DRS_feature_47', 'DRS_feature_48', 'DRS_feature_49', 'DRS_feature_50', 'DRS_feature_51', 'DRS_feature_52', 'DRS_feature_53', 'DRS_fe

,DRS_feature_1,DRS_feature_2,DRS_feature_3,DRS_feature_4,DRS_feature_5,DRS_feature_6,DRS_feature_7,DRS_feature_8,DRS_feature_9,DRS_feature_10,...,DRS_feature_325,DRS_feature_326,DRS_feature_327,DRS_feature_328,DRS_feature_329,DRS_feature_330,synergy_score,drugname1,drugname2,cellline
0,-0.515973,0.370004,0.225882,0.385421,0.089884,0.208102,0.083458,0.333762,-0.384937,-0.075529,...,0.185562,-0.418298,0.021533,-0.057358,-0.064901,0.027048,20.0,CCC1(O)CC2CN(CCc3c([nH]c4ccccc34)C(C(=O)OC)(c3...,COc1cc2c(Nc3ccc(Br)cc3F)ncnc2cc1OCC1CCN(C)CC1,ACH-001190
1,-0.515973,0.370004,0.225882,0.385421,0.089884,0.208102,0.083458,0.333762,-0.384937,-0.075529,...,0.165448,0.304612,-0.019375,0.069499,0.049702,-0.022915,-3.0,CCC1(O)CC2CN(CCc3c([nH]c4ccccc34)C(C(=O)OC)(c3...,CC(Oc1cc(-c2cnn(C3CCNCC3)c2)cnc1N)c1c(Cl)ccc(F...,ACH-001190
2,-0.515973,0.370004,0.225882,0.385421,0.089884,0.208102,0.083458,0.333762,-0.384937,-0.075529,...,-0.135688,0.375762,-0.024565,-0.061268,-0.052014,0.029371,-38.0,CCC1(O)CC2CN(CCc3c([nH]c4ccccc34)C(C(=O)OC)(c3...,O=C(O)CCCc1ccc(N(CCCl)CCCl)cc1,ACH-001190
3,-0.515973,0.370004,0.225882,0.385421,0.089884,0.208102,0.083458,0.333762,-0.384937,-0.075529,...,0.144554,0.304637,0.018489,0.052924,0.045733,-0.026488,-42.0,CCC1(O)CC2CN(CCc3c([nH]c4ccccc34)C(C(=O)OC)(c3...,CN(C)NN=C1N=CN=C1C(N)=O,ACH-001190
4,-0.515973,0.370004,0.225882,0.385421,0.089884,0.208102,0.083458,0.333762,-0.384937,-0.075529,...,-0.160220,0.321535,0.024040,-0.080988,0.055648,0.045245,22.0,CCC1(O)CC2CN(CCc3c([nH]c4ccccc34)C(C(=O)OC)(c3...,CCCS(=O)(=O)Nc1ccc(F)c(C(=O)c2c[nH]c3ncc(-c4cc...,ACH-001190



DATASET: DrugComb
Shape: (17139, 334)
Columns: ['DRS_feature_1', 'DRS_feature_2', 'DRS_feature_3', 'DRS_feature_4', 'DRS_feature_5', 'DRS_feature_6', 'DRS_feature_7', 'DRS_feature_8', 'DRS_feature_9', 'DRS_feature_10', 'DRS_feature_11', 'DRS_feature_12', 'DRS_feature_13', 'DRS_feature_14', 'DRS_feature_15', 'DRS_feature_16', 'DRS_feature_17', 'DRS_feature_18', 'DRS_feature_19', 'DRS_feature_20', 'DRS_feature_21', 'DRS_feature_22', 'DRS_feature_23', 'DRS_feature_24', 'DRS_feature_25', 'DRS_feature_26', 'DRS_feature_27', 'DRS_feature_28', 'DRS_feature_29', 'DRS_feature_30', 'DRS_feature_31', 'DRS_feature_32', 'DRS_feature_33', 'DRS_feature_34', 'DRS_feature_35', 'DRS_feature_36', 'DRS_feature_37', 'DRS_feature_38', 'DRS_feature_39', 'DRS_feature_40', 'DRS_feature_41', 'DRS_feature_42', 'DRS_feature_43', 'DRS_feature_44', 'DRS_feature_45', 'DRS_feature_46', 'DRS_feature_47', 'DRS_feature_48', 'DRS_feature_49', 'DRS_feature_50', 'DRS_feature_51', 'DRS_feature_52', 'DRS_feature_53', 'DRS_f

,DRS_feature_1,DRS_feature_2,DRS_feature_3,DRS_feature_4,DRS_feature_5,DRS_feature_6,DRS_feature_7,DRS_feature_8,DRS_feature_9,DRS_feature_10,...,DRS_feature_325,DRS_feature_326,DRS_feature_327,DRS_feature_328,DRS_feature_329,DRS_feature_330,synergy_score,drugname1,drugname2,cellline
0,-0.963012,0.330196,0.415354,0.681243,0.126370,0.340780,0.074045,0.312945,-0.318466,-0.079270,...,-0.116881,0.290728,0.024108,0.080590,-0.048774,-0.039340,1.0500,CC(C)CC(NC(=O)C(Cc1ccccc1)NC(=O)c1cnccn1)B(O)O,Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C...,ACH-000788
1,-0.963012,0.330196,0.415354,0.681243,0.126370,0.340780,0.074045,0.312945,-0.318466,-0.079270,...,0.148777,0.409056,-0.023054,-0.087388,-0.051881,0.029507,12.8950,CC(C)CC(NC(=O)C(Cc1ccccc1)NC(=O)c1cnccn1)B(O)O,CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)...,ACH-000788
2,-0.963012,0.330196,0.415354,0.681243,0.126370,0.340780,0.074045,0.312945,-0.318466,-0.079270,...,-0.129041,-0.363015,-0.016358,-0.048125,0.038844,-0.029173,2.6475,CC(C)CC(NC(=O)C(Cc1ccccc1)NC(=O)c1cnccn1)B(O)O,CCC1(O)C(=O)OCc2c1cc1n(c2=O)Cc2cc3c(CN(C)C)c(O...,ACH-000788
3,-0.791005,0.461537,0.266557,-0.507952,-0.123388,-0.230464,0.161379,0.319390,0.278324,0.058296,...,0.148777,0.409056,-0.023054,-0.087388,-0.051881,0.029507,13.4575,Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C...,CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)...,ACH-000788
4,-0.791005,0.461537,0.266557,-0.507952,-0.123388,-0.230464,0.161379,0.319390,0.278324,0.058296,...,-0.129041,-0.363015,-0.016358,-0.048125,0.038844,-0.029173,2.4525,Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C...,CCC1(O)C(=O)OCc2c1cc1n(c2=O)Cc2cc3c(CN(C)C)c(O...,ACH-000788



DATASET: DrugCombDB
Shape: (17139, 334)
Columns: ['DRS_feature_1', 'DRS_feature_2', 'DRS_feature_3', 'DRS_feature_4', 'DRS_feature_5', 'DRS_feature_6', 'DRS_feature_7', 'DRS_feature_8', 'DRS_feature_9', 'DRS_feature_10', 'DRS_feature_11', 'DRS_feature_12', 'DRS_feature_13', 'DRS_feature_14', 'DRS_feature_15', 'DRS_feature_16', 'DRS_feature_17', 'DRS_feature_18', 'DRS_feature_19', 'DRS_feature_20', 'DRS_feature_21', 'DRS_feature_22', 'DRS_feature_23', 'DRS_feature_24', 'DRS_feature_25', 'DRS_feature_26', 'DRS_feature_27', 'DRS_feature_28', 'DRS_feature_29', 'DRS_feature_30', 'DRS_feature_31', 'DRS_feature_32', 'DRS_feature_33', 'DRS_feature_34', 'DRS_feature_35', 'DRS_feature_36', 'DRS_feature_37', 'DRS_feature_38', 'DRS_feature_39', 'DRS_feature_40', 'DRS_feature_41', 'DRS_feature_42', 'DRS_feature_43', 'DRS_feature_44', 'DRS_feature_45', 'DRS_feature_46', 'DRS_feature_47', 'DRS_feature_48', 'DRS_feature_49', 'DRS_feature_50', 'DRS_feature_51', 'DRS_feature_52', 'DRS_feature_53', 'DRS

,DRS_feature_1,DRS_feature_2,DRS_feature_3,DRS_feature_4,DRS_feature_5,DRS_feature_6,DRS_feature_7,DRS_feature_8,DRS_feature_9,DRS_feature_10,...,DRS_feature_325,DRS_feature_326,DRS_feature_327,DRS_feature_328,DRS_feature_329,DRS_feature_330,synergy_score,drugname1,drugname2,cellline
0,-0.963012,0.330196,0.415354,0.681243,0.126370,0.340780,0.074045,0.312945,-0.318466,-0.079270,...,-0.116881,0.290728,0.024108,0.080590,-0.048774,-0.039340,1.0500,CC(C)CC(NC(=O)C(Cc1ccccc1)NC(=O)c1cnccn1)B(O)O,Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C...,ACH-000788
1,-0.963012,0.330196,0.415354,0.681243,0.126370,0.340780,0.074045,0.312945,-0.318466,-0.079270,...,0.148777,0.409056,-0.023054,-0.087388,-0.051881,0.029507,12.8950,CC(C)CC(NC(=O)C(Cc1ccccc1)NC(=O)c1cnccn1)B(O)O,CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)...,ACH-000788
2,-0.963012,0.330196,0.415354,0.681243,0.126370,0.340780,0.074045,0.312945,-0.318466,-0.079270,...,-0.129041,-0.363015,-0.016358,-0.048125,0.038844,-0.029173,2.6475,CC(C)CC(NC(=O)C(Cc1ccccc1)NC(=O)c1cnccn1)B(O)O,CCC1(O)C(=O)OCc2c1cc1n(c2=O)Cc2cc3c(CN(C)C)c(O...,ACH-000788
3,-0.791005,0.461537,0.266557,-0.507952,-0.123388,-0.230464,0.161379,0.319390,0.278324,0.058296,...,0.148777,0.409056,-0.023054,-0.087388,-0.051881,0.029507,13.4575,Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C...,CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)...,ACH-000788
4,-0.791005,0.461537,0.266557,-0.507952,-0.123388,-0.230464,0.161379,0.319390,0.278324,0.058296,...,-0.129041,-0.363015,-0.016358,-0.048125,0.038844,-0.029173,2.4525,Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C...,CCC1(O)C(=O)OCc2c1cc1n(c2=O)Cc2cc3c(CN(C)C)c(O...,ACH-000788



DATASET: OncologyScreen
Shape: (1160, 334)
Columns: ['DRS_feature_1', 'DRS_feature_2', 'DRS_feature_3', 'DRS_feature_4', 'DRS_feature_5', 'DRS_feature_6', 'DRS_feature_7', 'DRS_feature_8', 'DRS_feature_9', 'DRS_feature_10', 'DRS_feature_11', 'DRS_feature_12', 'DRS_feature_13', 'DRS_feature_14', 'DRS_feature_15', 'DRS_feature_16', 'DRS_feature_17', 'DRS_feature_18', 'DRS_feature_19', 'DRS_feature_20', 'DRS_feature_21', 'DRS_feature_22', 'DRS_feature_23', 'DRS_feature_24', 'DRS_feature_25', 'DRS_feature_26', 'DRS_feature_27', 'DRS_feature_28', 'DRS_feature_29', 'DRS_feature_30', 'DRS_feature_31', 'DRS_feature_32', 'DRS_feature_33', 'DRS_feature_34', 'DRS_feature_35', 'DRS_feature_36', 'DRS_feature_37', 'DRS_feature_38', 'DRS_feature_39', 'DRS_feature_40', 'DRS_feature_41', 'DRS_feature_42', 'DRS_feature_43', 'DRS_feature_44', 'DRS_feature_45', 'DRS_feature_46', 'DRS_feature_47', 'DRS_feature_48', 'DRS_feature_49', 'DRS_feature_50', 'DRS_feature_51', 'DRS_feature_52', 'DRS_feature_53', '

,DRS_feature_1,DRS_feature_2,DRS_feature_3,DRS_feature_4,DRS_feature_5,DRS_feature_6,DRS_feature_7,DRS_feature_8,DRS_feature_9,DRS_feature_10,...,DRS_feature_325,DRS_feature_326,DRS_feature_327,DRS_feature_328,DRS_feature_329,DRS_feature_330,synergy_score,drugname1,drugname2,cellline
0,-0.963012,0.330196,0.415354,0.681243,0.12637,0.34078,0.074045,0.312945,-0.318466,-0.07927,...,-0.116881,0.290728,0.024108,0.08059,-0.048774,-0.03934,21.659654,CC(C)CC(NC(=O)C(Cc1ccccc1)NC(=O)c1cnccn1)B(O)O,Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C...,ACH-000788
1,-0.963012,0.330196,0.415354,0.681243,0.12637,0.34078,0.074045,0.312945,-0.318466,-0.07927,...,-0.116881,0.290728,0.024108,0.08059,-0.048774,-0.03934,27.353028,CC(C)CC(NC(=O)C(Cc1ccccc1)NC(=O)c1cnccn1)B(O)O,Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C...,ACH-000657
2,-0.963012,0.330196,0.415354,0.681243,0.12637,0.34078,0.074045,0.312945,-0.318466,-0.07927,...,-0.116881,0.290728,0.024108,0.08059,-0.048774,-0.03934,24.410276,CC(C)CC(NC(=O)C(Cc1ccccc1)NC(=O)c1cnccn1)B(O)O,Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C...,ACH-000219
3,-0.963012,0.330196,0.415354,0.681243,0.12637,0.34078,0.074045,0.312945,-0.318466,-0.07927,...,-0.116881,0.290728,0.024108,0.08059,-0.048774,-0.03934,-2.125864,CC(C)CC(NC(=O)C(Cc1ccccc1)NC(=O)c1cnccn1)B(O)O,Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C...,ACH-000713
4,-0.963012,0.330196,0.415354,0.681243,0.12637,0.34078,0.074045,0.312945,-0.318466,-0.07927,...,-0.116881,0.290728,0.024108,0.08059,-0.048774,-0.03934,13.463733,CC(C)CC(NC(=O)C(Cc1ccccc1)NC(=O)c1cnccn1)B(O)O,Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C...,ACH-001061



DATASET: Oneil
Shape: (2714, 334)
Columns: ['DRS_feature_1', 'DRS_feature_2', 'DRS_feature_3', 'DRS_feature_4', 'DRS_feature_5', 'DRS_feature_6', 'DRS_feature_7', 'DRS_feature_8', 'DRS_feature_9', 'DRS_feature_10', 'DRS_feature_11', 'DRS_feature_12', 'DRS_feature_13', 'DRS_feature_14', 'DRS_feature_15', 'DRS_feature_16', 'DRS_feature_17', 'DRS_feature_18', 'DRS_feature_19', 'DRS_feature_20', 'DRS_feature_21', 'DRS_feature_22', 'DRS_feature_23', 'DRS_feature_24', 'DRS_feature_25', 'DRS_feature_26', 'DRS_feature_27', 'DRS_feature_28', 'DRS_feature_29', 'DRS_feature_30', 'DRS_feature_31', 'DRS_feature_32', 'DRS_feature_33', 'DRS_feature_34', 'DRS_feature_35', 'DRS_feature_36', 'DRS_feature_37', 'DRS_feature_38', 'DRS_feature_39', 'DRS_feature_40', 'DRS_feature_41', 'DRS_feature_42', 'DRS_feature_43', 'DRS_feature_44', 'DRS_feature_45', 'DRS_feature_46', 'DRS_feature_47', 'DRS_feature_48', 'DRS_feature_49', 'DRS_feature_50', 'DRS_feature_51', 'DRS_feature_52', 'DRS_feature_53', 'DRS_featu

,DRS_feature_1,DRS_feature_2,DRS_feature_3,DRS_feature_4,DRS_feature_5,DRS_feature_6,DRS_feature_7,DRS_feature_8,DRS_feature_9,DRS_feature_10,...,DRS_feature_325,DRS_feature_326,DRS_feature_327,DRS_feature_328,DRS_feature_329,DRS_feature_330,synergy_score,drugname1,drugname2,cellline
0,-0.963012,0.330196,0.415354,0.681243,0.12637,0.34078,0.074045,0.312945,-0.318466,-0.07927,...,-0.116881,0.290728,0.024108,0.08059,-0.048774,-0.03934,21.659654,CC(C)CC(NC(=O)C(Cc1ccccc1)NC(=O)c1cnccn1)B(O)O,Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C...,ACH-000788
1,-0.963012,0.330196,0.415354,0.681243,0.12637,0.34078,0.074045,0.312945,-0.318466,-0.07927,...,-0.116881,0.290728,0.024108,0.08059,-0.048774,-0.03934,27.353028,CC(C)CC(NC(=O)C(Cc1ccccc1)NC(=O)c1cnccn1)B(O)O,Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C...,ACH-000657
2,-0.963012,0.330196,0.415354,0.681243,0.12637,0.34078,0.074045,0.312945,-0.318466,-0.07927,...,-0.116881,0.290728,0.024108,0.08059,-0.048774,-0.03934,24.410276,CC(C)CC(NC(=O)C(Cc1ccccc1)NC(=O)c1cnccn1)B(O)O,Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C...,ACH-000219
3,-0.963012,0.330196,0.415354,0.681243,0.12637,0.34078,0.074045,0.312945,-0.318466,-0.07927,...,-0.116881,0.290728,0.024108,0.08059,-0.048774,-0.03934,15.328535,CC(C)CC(NC(=O)C(Cc1ccccc1)NC(=O)c1cnccn1)B(O)O,Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C...,ACH-000757
4,-0.963012,0.330196,0.415354,0.681243,0.12637,0.34078,0.074045,0.312945,-0.318466,-0.07927,...,-0.116881,0.290728,0.024108,0.08059,-0.048774,-0.03934,-2.125864,CC(C)CC(NC(=O)C(Cc1ccccc1)NC(=O)c1cnccn1)B(O)O,Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C...,ACH-000713


In [17]:
def train_random_forest(df, dataset_name):

    # ----------------------------------
    # 1. Identify required columns
    # ----------------------------------

    required_columns = [
        "drugname1",
        "drugname2",
        "cellline",
        "synergy_score"
    ]

    for col in required_columns:

        if col not in df.columns:
            raise ValueError(f"Missing column: {col}")

    # Identify DRS features

    drs_columns = [
        col for col in df.columns
        if col.startswith("DRS_feature_")
    ]

    if not drs_columns:
        raise ValueError("No DRS features found!")

    print("DRS features:", len(drs_columns))


    # ----------------------------------
    # 2. Clean synergy score
    # ----------------------------------

    df["synergy_score"] = pd.to_numeric(
        df["synergy_score"],
        errors="coerce"
    )

    df = df.dropna(
        subset=["synergy_score"]
    ).copy()

    df = df[
        np.isfinite(df["synergy_score"])
    ].copy()


    # ----------------------------------
    # 3. Generate Morgan fingerprints
    # ----------------------------------

    drug1_fp = []
    drug2_fp = []
    valid_indices = []

    for index, row in df.iterrows():

        fp1 = generate_fingerprint(row["drugname1"])

        fp2 = generate_fingerprint(row["drugname2"])

        if fp1 is not None and fp2 is not None:

            drug1_fp.append(fp1)
            drug2_fp.append(fp2)

            valid_indices.append(index)

    invalid_count = len(df) - len(valid_indices)

    df = df.loc[valid_indices].copy()

    drug1_fp = np.array(drug1_fp)
    drug2_fp = np.array(drug2_fp)

    print("Valid combinations:", len(df))
    print("Invalid combinations:", invalid_count)

    if len(df) < 2:
        raise ValueError("Insufficient valid drug combinations!")


    # ----------------------------------
    # 4. Create Drug 1 fingerprint columns
    # ----------------------------------

    drug1_df = pd.DataFrame(
        drug1_fp,
        columns=[
            f"Drug1_FP_{i}" for i in range(2048)
        ],
        index=df.index
    )


    # ----------------------------------
    # 5. Create Drug 2 fingerprint columns
    # ----------------------------------

    drug2_df = pd.DataFrame(
        drug2_fp,
        columns=[
            f"Drug2_FP_{i}" for i in range(2048)
        ],
        index=df.index
    )


    # ----------------------------------
    # 6. Process DRS features
    # ----------------------------------

    drs_data = df[drs_columns].apply(
        pd.to_numeric,
        errors="coerce"
    )

    drs_data = drs_data.replace(
        [np.inf, -np.inf],
        np.nan
    )


    # ----------------------------------
    # 7. Encode cell-line information
    # ----------------------------------

    cellline_data = pd.get_dummies(
        df["cellline"].fillna("Unknown").astype(str),
        prefix="cellline",
        dtype=int
    )


    # ----------------------------------
    # 8. Combine input features
    # ----------------------------------

    X = pd.concat(
        [
            drug1_df,
            drug2_df,
            drs_data,
            cellline_data
        ],
        axis=1
    )

    y = df["synergy_score"]

    print("Total input features:", X.shape[1])


    # ----------------------------------
    # 9. Split training and testing data
    # ----------------------------------

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )


    # ----------------------------------
    # 10. Handle missing DRS values
    # ----------------------------------

    # Calculate medians using training data only

    medians = X_train[drs_columns].median().fillna(0)

    X_train.loc[:, drs_columns] = (
        X_train[drs_columns].fillna(medians)
    )

    X_test.loc[:, drs_columns] = (
        X_test[drs_columns].fillna(medians)
    )


    # ----------------------------------
    # 11. Create Random Forest model
    # ----------------------------------

    rf_model = RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        max_features=1.0
    )


    # ----------------------------------
    # 12. Train model
    # ----------------------------------

    rf_model.fit(X_train, y_train)

    print("Model training completed!")


    # ----------------------------------
    # 13. Make predictions
    # ----------------------------------

    y_pred = rf_model.predict(X_test)


    # ----------------------------------
    # 14. Calculate evaluation metrics
    # ----------------------------------

    mae = mean_absolute_error(y_test, y_pred)

    mse = mean_squared_error(y_test, y_pred)

    rmse = np.sqrt(mse)

    r2 = r2_score(y_test, y_pred)


    # ----------------------------------
    # 15. Return results
    # ----------------------------------

    return {
        "Dataset": dataset_name,
        "Samples": len(df),
        "Features": X.shape[1],
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2 Score": r2
    }

In [18]:
all_results = []

for name, path in datasets.items():

    print("\n" + "=" * 65)
    print("TRAINING DATASET:", name)
    print("=" * 65)

    try:

        # Load dataset
        df = pd.read_csv(path)

        print("Original shape:", df.shape)

        # Train Random Forest
        result = train_random_forest(df, name)

        # Store results
        all_results.append(result)

        # Display individual performance

        print("\nMODEL PERFORMANCE:", name)

        print("-" * 45)

        print("MAE     :", round(result["MAE"], 4))

        print("MSE     :", round(result["MSE"], 4))

        print("RMSE    :", round(result["RMSE"], 4))

        print("R2 Score:", round(result["R2 Score"], 4))

        print("\nTraining completed successfully!")

    except Exception as e:

        print("\nERROR in", name, ":", str(e))


print("\nALL DATASET TRAINING PROCESSES COMPLETED!")


TRAINING DATASET: ALMANAC
Original shape: (29251, 334)
DRS features: 330
Valid combinations: 29251
Invalid combinations: 0
Total input features: 4465
Model training completed!

MODEL PERFORMANCE: ALMANAC
---------------------------------------------
MAE     : 29.5318
MSE     : 1613.1937
RMSE    : 40.1646
R2 Score: 0.4828

Training completed successfully!

TRAINING DATASET: DrugComb
Original shape: (17139, 334)
DRS features: 330
Valid combinations: 17139
Invalid combinations: 0
Total input features: 4491
Model training completed!

MODEL PERFORMANCE: DrugComb
---------------------------------------------
MAE     : 2.7378
MSE     : 15.1559
RMSE    : 3.8931
R2 Score: 0.4747

Training completed successfully!

TRAINING DATASET: DrugCombDB
Original shape: (17139, 334)
DRS features: 330
Valid combinations: 17139
Invalid combinations: 0
Total input features: 4491
Model training completed!

MODEL PERFORMANCE: DrugCombDB
---------------------------------------------
MAE     : 2.7378
MSE     : 15

In [19]:
results_df = pd.DataFrame(all_results)

print("\n")
print("=" * 85)
print("FINAL RANDOM FOREST PERFORMANCE COMPARISON")
print("=" * 85)

if not results_df.empty:

    display(
        results_df.round(4)
    )

else:

    print("No datasets were successfully trained.")



FINAL RANDOM FOREST PERFORMANCE COMPARISON


,Dataset,Samples,Features,MAE,MSE,RMSE,R2 Score
0,ALMANAC,29251,4465,29.5318,1613.1937,40.1646,0.4828
1,DrugComb,17139,4491,2.7378,15.1559,3.8931,0.4747
2,DrugCombDB,17139,4491,2.7378,15.1559,3.8931,0.4747
3,OncologyScreen,1160,4455,13.3519,369.8771,19.2322,0.3622
4,Oneil,2714,4458,11.6013,257.7565,16.0548,0.4853
